# Basic-Lite TRACE — Fase 5 en TPU v5e-8 (BF16 nativo + `torch_xla`)

**Copyright © 2026 Miguel19x. Todos los derechos reservados (`LICENSE.txt`).**

Este notebook está diseñado para ejecutarse en **Kaggle TPU v5e-8 (`PJRT_DEVICE=TPU`) en modo single-chip (1 chip TPU v5e, 16 GiB HBM, BF16 nativo)** sin sobrecarga de DDP multi-chip, respetando el límite de sesión de 9 horas (`--max-wall-hours 8.0`):
1. Instala `transformers>=4.51.0`, `peft`, `accelerate` y `safetensors` sin `--upgrade` (para preservar `torch` y `torch_xla` preinstalados en Kaggle), verifica el dispositivo TPU en un **subproceso efímero** (liberando `libtpu_lockfile` antes del entrenamiento), y clona/actualiza `Basic-Lite-TRACE`.
2. Ejecuta los tests unitarios de Fase 5 y TPU/XLA (`tests/test_phase5_reasoning.py` y `tests/test_phase5_tpu_xla.py`) antes de consumir cuota TPU.
3. Ajusta `microsoft/bitnet-b1.58-2B-4T-bf16` en **BF16 nativo** con **LoRA (`r=16`) + QAT-STE (W1.58A8) + SFT con padding estático + GRPO/RLVR corto** (`--device xla --max-wall-hours 8.0`).
4. Exporta el modelo fusionado a **`Paged BitCOS v1.1`** (`tools/import_bitnet.py`) listo para descargar y evaluar en CPU local (`i3-1115G4`, `OMP_NUM_THREADS=2`).

In [ ]:
import os

# Congela la versión preinstalada de `torch` para que `peft`/`accelerate` no rompan la ABI de `torch_xla`
!python3 -c "import torch; open('/tmp/torch_constraint.txt','w').write(f'torch=={torch.__version__.split(\"+\")[0]}\n')"
!pip -q install -c /tmp/torch_constraint.txt "transformers>=4.51.0" peft accelerate safetensors

# Sonda TPU en subproceso efímero: libera `libtpu_lockfile` antes de lanzar `train_phase5_sft_grpo.py`
!PJRT_DEVICE=TPU python3 -c "import torch, torch_xla, torch_xla.core.xla_model as xm; print('PyTorch:', torch.__version__, 'torch_xla:', torch_xla.__version__, 'XLA device:', xm.xla_device())"

REPO_DIR = "/kaggle/working/Basic-Lite-TRACE"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Miguel19x/Basic-Lite-TRACE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 1. Verificar tests unitarios de Fase 5 + TPU/XLA antes de lanzar el entrenamiento
!OMP_NUM_THREADS=2 PYTHONPATH=. pytest tests/test_phase5_reasoning.py tests/test_phase5_tpu_xla.py -v

In [ ]:
# 2. Entrenamiento BF16 nativo + QAT-STE (W1.58A8) SFT (<think> + TIR) + GRPO corto en TPU v5e-8
!PJRT_DEVICE=TPU PYTHONPATH=. python3 scripts/train_phase5_sft_grpo.py \
    --backend hf-2b \
    --device xla \
    --max-wall-hours 8.0 \
    --model-id microsoft/bitnet-b1.58-2B-4T-bf16 \
    --use-lora --qat-ste --lora-r 16 --lora-alpha 32 \
    --num-math 360 --num-code 180 --num-factual 120 --general-ratio 0.15 \
    --sft-steps 800 --grad-accum 8 --lr 1e-3 --max-seq-len 384 \
    --grpo-steps 24 --grpo-group-size 4 --grpo-max-new-tokens 96 \
    --export-dataset /kaggle/working/fase5_tir_dataset_tpu.jsonl \
    --out-dir /kaggle/working/bitnet_2b_phase5_tpu_hf

In [ ]:
# 3. Exportar el modelo fusionado a Paged BitCOS v1.1 (W1.58 + embeddings FP16) + TRTK v2
!PYTHONPATH=. python3 tools/import_bitnet.py \
    --model-dir /kaggle/working/bitnet_2b_phase5_tpu_hf \
    --output-model /kaggle/working/bitnet_2b_phase5_tpu_fp16.bin \
    --output-tokenizer /kaggle/working/bitnet_2b_phase5_tpu_tok_v2.bin \
    --embedding-dtype float16
!ls -lh /kaggle/working/bitnet_2b_phase5_tpu_fp16.bin /kaggle/working/bitnet_2b_phase5_tpu_tok_v2.bin

## Evaluación local en `i3-1115G4` (Puerta de Salida de Fase 5)

Descarga `bitnet_2b_phase5_tpu_fp16.bin` y `bitnet_2b_phase5_tpu_tok_v2.bin` a `checkpoints/` en tu equipo local y ejecuta:
```bash
OMP_NUM_THREADS=2 PYTHONPATH=. venv/bin/python scripts/run_bench_4b.py \
    --model checkpoints/bitnet_2b_phase5_tpu_fp16.bin \
    --tokenizer checkpoints/bitnet_2b_phase5_tpu_tok_v2.bin \
    --max-new-tokens 128 \
    --json-out results/fase5_bench_2b_tpu_c1_c3.json
```